In [1]:
import pandas as pd
from skrub import TableReport

df = pd.read_csv('../resources/movetimes3481887positions-withWasCapture-withMoveNumber-withoutPieces.csv')
TableReport(df)

Processing column   9 / 9


,,,,,,,,,


In [2]:
from sklearn.model_selection import train_test_split

nontest, test = train_test_split(df, test_size = 0.1, random_state = 42)

train, val = train_test_split(nontest, test_size = 0.1, random_state = 42)
# given the size of the datatest, cross-validation is unnecessary to maintain a large training dataset.

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2820328 entries, 744012 to 3336675
Data columns (total 9 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   was_capture                       bool   
 1   move_number                       int64  
 2   initial_time                      int64  
 3   increment                         int64  
 4   time_remaining_for_this_position  float64
 5   time_opponent_took                float64
 6   number_of_legal_moves             int64  
 7   in_check                          bool   
 8   time_taken_for_this_position      float64
dtypes: bool(2), float64(3), int64(4)
memory usage: 177.5 MB


In [4]:
val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 313370 entries, 489006 to 962524
Data columns (total 9 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   was_capture                       313370 non-null  bool   
 1   move_number                       313370 non-null  int64  
 2   initial_time                      313370 non-null  int64  
 3   increment                         313370 non-null  int64  
 4   time_remaining_for_this_position  313370 non-null  float64
 5   time_opponent_took                313370 non-null  float64
 6   number_of_legal_moves             313370 non-null  int64  
 7   in_check                          313370 non-null  bool   
 8   time_taken_for_this_position      313370 non-null  float64
dtypes: bool(2), float64(3), int64(4)
memory usage: 19.7 MB


In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 348189 entries, 3363431 to 298423
Data columns (total 9 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   was_capture                       348189 non-null  bool   
 1   move_number                       348189 non-null  int64  
 2   initial_time                      348189 non-null  int64  
 3   increment                         348189 non-null  int64  
 4   time_remaining_for_this_position  348189 non-null  float64
 5   time_opponent_took                348189 non-null  float64
 6   number_of_legal_moves             348189 non-null  int64  
 7   in_check                          348189 non-null  bool   
 8   time_taken_for_this_position      348189 non-null  float64
dtypes: bool(2), float64(3), int64(4)
memory usage: 21.9 MB


In [6]:
from sklearn.dummy import DummyRegressor
from sklearn import linear_model

dummy_regression_mean = DummyRegressor(strategy="mean")
dummy_regression_median = DummyRegressor(strategy="median")
linear_regression_model_one = linear_model.LinearRegression()

# 'time_taken_for_this_position' is the target feature, so we'll take it out and put it in a variables called y
def X_y_split(dataframe):
    X = dataframe.drop('time_taken_for_this_position', axis=1)
    y = dataframe['time_taken_for_this_position']
    return (X, y)

train_X, train_y = X_y_split(train)
val_X, val_y = X_y_split(val)

In [7]:
from sklearn.metrics import mean_squared_error

dummy_regression_mean.fit(train_X, train_y)
dummy_regression_median.fit(train_X, train_y)
linear_regression_model_one.fit(train_X, train_y)

print('dummy_regression_mean score on validation set:', mean_squared_error(val_y, dummy_regression_mean.predict(val_X)))
print('dummy_regression_median score on validation set:', mean_squared_error(val_y, dummy_regression_median.predict(val_X)))
print('First linear regression model score on validation set:', mean_squared_error(val_y, linear_regression_model_one.predict(val_X)))
# lower mean_squared_error is better

dummy_regression_mean score on validation set: 73.32519935919755
dummy_regression_median score on validation set: 80.64002901937006
First linear regression model score on validation set: 56.80659841369981


In [8]:
print('features and their linear regression coefficients')
coefficients = linear_regression_model_one.coef_
index = 0
for column in train_X:
    print(column, coefficients[index])
    index += 1

print('intercept:', linear_regression_model_one.intercept_)

features and their linear regression coefficients
was_capture -0.8854172992052682
move_number -0.05103132495214151
initial_time 0.021655853892468845
increment 0.39717018495024825
time_remaining_for_this_position -0.015086531224241272
time_opponent_took 0.19298039929487032
number_of_legal_moves 0.10434049571210303
in_check 1.0943433402829832
intercept: -0.9682846035325552


In [9]:
# To check the distribution of predictions from our first linear regression model
TableReport(pd.DataFrame(linear_regression_model_one.predict(val_X)))

Processing column   1 / 1


<TableReport: use .open() to display>

In [10]:
# We can see that the median value of our target (time_taken_for_this_position in df) is 2.24 seconds,
# while the mean is 4.96 seconds (quite different). The distribution is skewed and has outliers.

# Our resulting linear regression model outputs a mean of 4.96 seconds (identical to the ground truth)
# but has a median of 4.09 seconds (rather than 2.24 seconds).
# So our model might be impacted by outliers.

# Thus, we will remove outliers from our training data (but not validation or test)
# as we don't want our model to output move times larger than is common just because the mean is skewed by outliers.

In [11]:
Q1 = train_y.quantile(0.25)
Q3 = train_y.quantile(0.75)
IQR = Q3 - Q1

mask = train_y <= (Q3 + 1.5 * IQR)
train_X = train_X[mask]
train_y = train_y[mask]


# We'll also remove samples where time_opponent_took are outliers. 
# This is because this is a similar kind of feature to the target.
Q1 = train_X['time_opponent_took'].quantile(0.25)
Q3 = train_X['time_opponent_took'].quantile(0.75)
IQR = Q3 - Q1

mask = train_X['time_opponent_took'] <= (Q3 + 1.5 * IQR)
train_X = train_X[mask]
train_y = train_y[mask]


In [12]:
TableReport(train_X)

Processing column   8 / 8


<TableReport: use .open() to display>

In [13]:
TableReport(train_y)

Processing column   1 / 1


<TableReport: use .open() to display>

In [14]:
linear_regression_model_two = linear_model.LinearRegression()
linear_regression_model_two.fit(train_X, train_y)
print('Second linear regression model score on validation set:', mean_squared_error(val_y, linear_regression_model_two.predict(val_X)))
# The score will be worse (higher error) as the validation data has outliers included, but we don't want to model outliers.

Second linear regression model score on validation set: 62.90666908712581


In [15]:
print('features and their linear regression coefficients')
coefficients = linear_regression_model_two.coef_
index = 0
for column in train_X:
    print(column, coefficients[index])
    index += 1

print('intercept:', linear_regression_model_two.intercept_)

features and their linear regression coefficients
was_capture -0.32108144019064794
move_number -0.009190687934514599
initial_time 0.005927153262152473
increment 0.06491330501699785
time_remaining_for_this_position -0.0032424802518566703
time_opponent_took 0.2631837208500079
number_of_legal_moves 0.0380929003850027
in_check 0.5717891514475931
intercept: 0.36686831532547126


In [16]:
# To check the distribution of predictions from our outlier-removed linear regression model
TableReport(pd.DataFrame(linear_regression_model_two.predict(val_X)))

Processing column   1 / 1


<TableReport: use .open() to display>

In [17]:
# The distribution of predicted move times is looking better after removing outliers.
# There are also no longer a significant amount of negative predictions 
# and they're not as extreme (-2.08 is the lowest prediction here instead of -12).
# Median is now 2.77 seconds, closer to actual median move times.

# Let's try using domain knowledge to create some engineered features that might improve the regression model.

In [18]:
import numpy as np
import skrub

# A feature to represent people hurrying up when they're low on time. 
# This feature is largest when there's nearly no time remaining,
# and gets smaller as there's more remaining time.
df['time_pressure'] = 1 / (1 + df['time_remaining_for_this_position'])

# Features to represent players no longer being in familiar territory (going "out of book"). 
# These features are sigmoid functions. They're roughly 0 before a turning point, and then transition to 1 after.
# Multiple features will be made with different turning points to see which move number people actually tend to go out-of-book.
df['out_of_book_depth_4'] = 1 / (1 + np.exp(4 - df['move_number']))
df['out_of_book_depth_5'] = 1 / (1 + np.exp(5 - df['move_number']))
df['out_of_book_depth_6'] = 1 / (1 + np.exp(6 - df['move_number']))
df['out_of_book_depth_7'] = 1 / (1 + np.exp(7 - df['move_number']))
df['out_of_book_depth_8'] = 1 / (1 + np.exp(8 - df['move_number']))
df['out_of_book_depth_9'] = 1 / (1 + np.exp(9 - df['move_number']))

target = 'time_taken_for_this_position'
associations = skrub.column_associations(df)

# To see which out_of_book feature has best association with target feature
target_associations = associations[
    (associations['left_column_name'] == target) | 
    (associations['right_column_name'] == target)
]

In [19]:
target_associations.sort_values('pearson_corr', ascending=False)

,left_column_name,left_column_idx,right_column_name,right_column_idx,cramer_v,pearson_corr
60,initial_time,2,time_taken_for_this_position,8,0.054022,0.377734
53,time_opponent_took,5,time_taken_for_this_position,8,0.073236,0.344006
52,time_remaining_for_this_position,4,time_taken_for_this_position,8,0.077884,0.319203
63,increment,3,time_taken_for_this_position,8,0.027695,0.259601
80,number_of_legal_moves,6,time_taken_for_this_position,8,0.005171,0.139693
85,time_taken_for_this_position,8,out_of_book_depth_6,12,0.004229,0.094256
84,time_taken_for_this_position,8,out_of_book_depth_7,13,0.004273,0.093334
86,time_taken_for_this_position,8,out_of_book_depth_5,11,0.003999,0.092791
83,time_taken_for_this_position,8,out_of_book_depth_8,14,0.004345,0.090178
87,time_taken_for_this_position,8,out_of_book_depth_4,10,0.003798,0.089005


In [20]:
# out_of_book_depth_6 has the best pearson_corr with the target feature. 
# time_pressure correlates well with the target feature
# (it's the absolute value of pearson_corr that indicates good correlation),
# These two engineered features will be added to our linear regression classifier

In [21]:
# "out_of_book_depth_6" is a poor name. Using "opening_unfamiliarity" as the name instead.

def use_engineered_features(dataframe):
    dataframe['time_pressure'] = 1 / (1 + dataframe['time_remaining_for_this_position'])
    dataframe['opening_unfamiliarity'] = 1 / (1 + np.exp(6 - dataframe['move_number']))  

use_engineered_features(train_X)
use_engineered_features(val_X)

In [22]:
# To check that we've replace our old features with the engineered ones
TableReport(train_X)

Processing column  10 / 10


,,,,,,,,,,


In [23]:
linear_regression_model_three = linear_model.LinearRegression()
linear_regression_model_three.fit(train_X, train_y)
print('Third linear regression model score on validation set:', mean_squared_error(val_y, linear_regression_model_three.predict(val_X)))

Third linear regression model score on validation set: 62.68963848239552


In [24]:
print('features and their linear regression coefficients')
coefficients = linear_regression_model_three.coef_
index = 0
for column in train_X:
    print(column, coefficients[index])
    index += 1

print('intercept:', linear_regression_model_three.intercept_)

features and their linear regression coefficients
was_capture -0.49985990582824846
move_number -0.022910755280864357
initial_time 0.005808197681903694
increment 0.0678994221188859
time_remaining_for_this_position -0.0031094473295077
time_opponent_took 0.2111450747447485
number_of_legal_moves 0.024391655328449374
in_check 0.22084024995566198
time_pressure -4.365343091495533
opening_unfamiliarity 1.5483814263122477
intercept: 0.07368122740219984


In [25]:
# To check the distribution of this engineered-features regression model
TableReport(pd.DataFrame(linear_regression_model_three.predict(val_X)))

Processing column   1 / 1


<TableReport: use .open() to display>

In [27]:
# Using the engineered features slightly improved validation performance (62.69 <  62.91)
# The high weights (coefficients) on the time_pressure and opening_unfamiliarity features are encouraging
# So the model used for the realistic move times feature will be a linear regression model including the two engineered features.
# Here are the coefficients:

# was_capture -0.49985990582824846
# move_number -0.022910755280864357
# initial_time 0.005808197681903694
# increment 0.0678994221188859
# time_remaining_for_this_position -0.0031094473295077
# time_opponent_took 0.2111450747447485
# number_of_legal_moves 0.024391655328449374
# in_check 0.22084024995566198
# time_pressure -4.365343091495533
# opening_unfamiliarity 1.5483814263122477
# intercept: 0.07368122740219984

# Where time_pressure = 1 / (1 + time_remaining_for_this_position)
# and opening_unfamiliarity = 1 / (1 + e ^ (6 - move_number))


# This model gets chosen for implementation at the poster fair,
# so we'll be evaluating it on the test set for it's final performance evaluation:
test_X, test_y = X_y_split(test)
use_engineered_features(test_X)

chosen_model_test_performance = mean_squared_error(test_y, linear_regression_model_three.predict(test_X))
print("Chosen model's mean squared error on test data:", chosen_model_test_performance)

Chosen model's mean squared error on test data: 64.39529010796556
